# Checkpoint 6 — K-Means vs. Behavioral Segmentation Evaluation
## AI-Powered E-commerce Customer Segmentation and Churn Analysis
### Brazilian E-Commerce Public Dataset by Olist

This notebook empirically evaluates whether **K-Means clustering** provides a viable, actionable customer segmentation compared to the **candidate rule-based behavioral segmentation** established in Checkpoint 5. Following `config/thresholds.yaml` pre-registered criteria (`candidate_k_min: 2`, `candidate_k_max: 6`, `require_business_interpretability: true`, and fallback `if_kmeans_fails: behavioral_segmentation`), we evaluate feature representations, clustering metrics, cluster RFM profiles, multi-seed stability, and commercial interpretability across all 93,358 delivered customers.

## 1. Objective & Methodological Scope

### Research Questions
1. How do candidate feature representations (raw RFM vs. log-transformed monetary) affect distance-based clustering?
2. How does K-Means perform across $K \in \{2, 3, 4, 5, 6\}$ in terms of inertia, silhouette scores, cluster size balance, and RFM centroids?
3. Is K-Means clustering stable across multiple random initializations (seeds 42, 123, 999)?
4. How does the 97.00% $F=1$ point mass affect the geometry of cluster centroids?
5. How does K-Means compare with the 9-cohort rule-based behavioral segmentation across granularity, repeat buyer differentiation, value concentration, and business interpretability?
6. Which approach should be approved as the primary customer segmentation methodology under the pre-registered decision gate?

> **Methodological Boundary**: Predictive churn modeling, train/test splitting, and the final selection between the 120-day and 180-day candidate churn windows remain strictly deferred.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.segmentation import run_segmentation_evaluation_audit

# Execute or load aggregate audit results
audit_data = run_segmentation_evaluation_audit(project_root=project_root)
meta = audit_data['population_metadata']

print(f"Checkpoint: {audit_data['checkpoint']}")
print(f"Audit UTC: {audit_data['audit_timestamp_utc']}")
print(f"Delivered orders analyzed: {meta['delivered_orders_analyzed']:,}")
print(f"Unique delivered customers: {meta['unique_customers_analyzed']:,}")
print(f"Reference timestamp: {meta['rfm_reference_timestamp']}")
print(f"Raw data immutability verified: {audit_data['raw_data_immutability_verified']}")


## 2. Feature Representation Evaluation

Distance-based clustering requires careful feature engineering. We examine feature skewness and correlations across raw RFM, log-transformed monetary values, and log-transformed recency.

In [ ]:
feat_eval = audit_data['feature_representation_evaluation']
skew_comp = feat_eval['skewness_comparison']
rep_assess = feat_eval['representation_assessments']

df_skew = pd.DataFrame([
    {'Feature Dimension': 'Recency (Days)', 'Raw Skewness': f"{skew_comp['Recency']:.4f}", 'Transformed Skewness': f"{skew_comp['log1p_Recency']:.4f} (log1p)", 'Assessment': 'Mild raw skewness; log1p over-compresses mid-range recency.'},
    {'Feature Dimension': 'Frequency (Orders)', 'Raw Skewness': f"{skew_comp['Frequency']:.4f}", 'Transformed Skewness': 'N/A (Discrete)', 'Assessment': 'Extreme skewness driven by 97.00% point mass at F=1.'},
    {'Feature Dimension': 'Monetary Value (BRL)', 'Raw Skewness': f"{skew_comp['Monetary']:.4f}", 'Transformed Skewness': f"{skew_comp['log1p_Monetary']:.4f} (log1p)", 'Assessment': 'Severe raw skewness (9.21) tamed to 0.53 by log1p transformation.'}
])

print(f"Selected Primary Candidate Representation: {feat_eval['selected_candidate_representation']}")
df_skew


## 3. K-Means Experiments Across K = 2..6

Systematic evaluation of K-Means clustering across candidate $K \in \{2, 3, 4, 5, 6\}$ using `StandardScaler([Recency, Frequency, log1p_Monetary])`. We evaluate inertia (elbow method), silhouette scores, and cluster size distributions.

In [ ]:
km_exp = audit_data['kmeans_experiments']['k_experiments']

k_rows = []
for k_key, k_info in km_exp.items():
    k_rows.append({
        'K (Clusters)': k_info['k'],
        'Inertia': f"{k_info['inertia']:,.1f}",
        'Silhouette Score (Sample)': f"{k_info['silhouette_score_sample']:.4f}",
        'Min Cluster Share (%)': f"{k_info['min_cluster_proportion_pct']:.2f}%",
        'Max Cluster Share (%)': f"{k_info['max_cluster_proportion_pct']:.2f}%",
        'Mean Multi-Seed ARI': f"{k_info['multi_seed_stability']['mean_ari']:.4f}",
        'Monolithic Repeat Cluster?': 'YES (All repeat in 1 cluster)' if k_info['has_monolithic_repeat_cluster'] else 'No'
    })

df_k_summary = pd.DataFrame(k_rows)
df_k_summary


## 4. The K=2 Silhouette Artifact & Monolithic Repeat Isolation

While $K=2$ yields the highest silhouette score (0.6883), inspection reveals that it merely splits the population into one-time buyers (97.00%) and repeat buyers (3.00%), providing zero actionable segmentation.

Furthermore, across $K=3, 4, 5, 6$, K-Means isolates all 2,801 repeat customers into a **single monolithic cluster**, completely failing to differentiate high-value champions from at-risk repeat buyers. We inspect cluster RFM profiles for $K=3$ and $K=4$.

In [ ]:
k3_prof = km_exp['k_3']['cluster_rfm_profiles']
k4_prof = km_exp['k_4']['cluster_rfm_profiles']

k4_rows = []
for c_id, prof in k4_prof.items():
    k4_rows.append({
        'Cluster': c_id,
        'Customer Count': f"{prof['count']:,}",
        'Share (%)': f"{prof['pct']:.2f}%",
        'Median Recency (d)': f"{prof['median_recency_days']:.1f}",
        'Mean Recency (d)': f"{prof['mean_recency_days']:.1f}",
        'Median Freq': f"{prof['median_frequency']:.0f}",
        'Mean Freq': f"{prof['mean_frequency']:.2f}",
        'Median Spend (BRL)': f"{prof['median_monetary_brl']:.2f}",
        'Mean Spend (BRL)': f"{prof['mean_monetary_brl']:.2f}"
    })

print("=== K-MEANS K=4 CLUSTER PROFILES ===")
df_k4_profiles = pd.DataFrame(k4_rows)
df_k4_profiles


## 5. Candidate Behavioral Customer Segmentation Evaluation

In contrast to K-Means, the 9-cohort rule-based behavioral segmentation partitions both repeat buyers (into 4 lifecycle stages: Champions, Loyal Regulars, At-Risk, Hibernating) and one-time buyers (into 5 recency/value tiers).

In [ ]:
b_seg = audit_data['behavioral_segmentation_evaluation']
b_prof = b_seg['segment_profiles']

b_rows = []
for seg_name, p in b_prof.items():
    b_rows.append({
        'Behavioral Cohort': seg_name,
        'Customer Count': f"{p['customer_count']:,}",
        'Share (%)': f"{p['customer_share_pct']:.2f}%",
        'Median Recency (d)': f"{p['median_recency_days']:.1f}",
        'Mean Freq': f"{p['mean_frequency']:.2f}",
        'Median Spend (BRL)': f"{p['median_observed_monetary_value_brl']:.2f}",
        'Total Spend (BRL)': f"{p['total_observed_monetary_value_brl']:,.2f}",
        'Monetary Share (%)': f"{p['monetary_value_share_pct']:.2f}%"
    })

df_behavioral = pd.DataFrame(b_rows).sort_values(by='Monetary Share (%)', ascending=False)
df_behavioral


## 6. Objective Comparative Benchmarking (7 Dimensions)

We compare K-Means clustering and rule-based behavioral segmentation side-by-side across 7 methodological and business criteria.

In [ ]:
comp_data = audit_data['methodological_comparison']['comparative_dimensions']

comp_rows = []
dim_titles = {
    'dimension_1_group_granularity': '1. Group Granularity',
    'dimension_2_repeat_customer_differentiation': '2. Repeat Customer Differentiation',
    'dimension_3_one_time_buyer_characterization': '3. One-Time Buyer Value Isolation',
    'dimension_4_cluster_size_balance': '4. Size Distribution Balance',
    'dimension_5_mathematical_stability': '5. Mathematical Stability',
    'dimension_6_business_interpretability': '6. Business Interpretability',
    'dimension_7_utility_for_retention_analysis': '7. Retention Analysis Utility'
}

for dim_key, dim_info in comp_data.items():
    comp_rows.append({
        'Evaluation Dimension': dim_titles.get(dim_key, dim_key),
        'K-Means Clustering Performance': dim_info['kmeans_evaluation'],
        'Rule-Based Behavioral Performance': dim_info['behavioral_evaluation'],
        'Evidence / Assessment': dim_info.get('evidence_assessment', dim_info.get('comparison_verdict', ''))
    })

df_comparison = pd.DataFrame(comp_rows)
print(f"Synthesis:\n{audit_data['methodological_comparison']['overall_synthesis']}\n")
df_comparison


## 7. Decision Gate & Pre-Registered Fallback Activation

Formulate the explicit methodological decisions in accordance with `config/thresholds.yaml`.

In [ ]:
dec = audit_data['methodological_decisions']

dec_rows = [
    {'Scope': 'K-Means Clustering', 'Status': dec['kmeans_clustering']['status'], 'Empirical Rationale': dec['kmeans_clustering']['reason']},
    {'Scope': 'Rule-Based Behavioral Segmentation', 'Status': dec['rule_based_behavioral_segmentation']['status'], 'Empirical Rationale': dec['rule_based_behavioral_segmentation']['reason']},
    {'Scope': 'Project-Level Pivot Fallback', 'Status': dec['project_fallback_status']['status'], 'Empirical Rationale': dec['project_fallback_status']['reason']},
    {'Scope': 'Churn Modeling & Window Selection', 'Status': dec['churn_modeling_and_window_selection']['status'], 'Empirical Rationale': dec['churn_modeling_and_window_selection']['reason']}
]

df_decisions = pd.DataFrame(dec_rows)
df_decisions


## 8. Checkpoint 6 Summary & Next Steps

### Key Empirical Findings
- **K-Means Evaluation**: K-Means clustering is structurally constrained by the 97.00% $F=1$ point mass. Across $K \in \{2..6\}$, it isolates all 2,801 repeat customers into a single monolithic cluster (zero lifecycle differentiation) and cuts one-time buyers along continuous Voronoi boundaries, failing the pre-registered `require_business_interpretability: true` gate. Status: **K-Means: NOT SUPPORTED**.
- **Behavioral Segmentation Evaluation**: The 9-cohort rule-based segmentation provides deterministic lifecycle cohorts, differentiating repeat customers into 4 recency/spend stages and one-time buyers into 5 cohorts (including 3,441 Recent High-Value VIPs generating 10.00% of total observed customer monetary value).
- **Fallback Mechanism**: In accordance with `fallbacks.segmentation.if_kmeans_fails: behavioral_segmentation`, the rule-based approach is **ADOPTED AS DOWNSTREAM FALLBACK CANDIDATE** for subsequent downstream retention/churn analysis where its practical usefulness can be evaluated further.
- **Project Immutability**: All 9 raw Olist CSV files retained identical SHA-256 hashes pre- and post-audit.

### Next Checkpoint
- Awaiting user review and authorization to proceed to **Phase 3: Predictive Churn Modeling & Retention Analysis**.